# Heckman 选择模型：选择性观测、排他性变量与贷款利率

> 本讲义用于《金融数据分析与建模》课程。配套文件 `06_heckman_codes_v2.ipynb` 用于生成 `./figs/` 中的配图和 `./data/heckman_credit_sim.csv`。

本章继续使用企业信贷背景，但研究对象从贷款金额转为贷款利率。这个变化非常关键：没有获得贷款的企业，其贷款金额可以记为 0；但它的贷款利率不是 0，而是不可观测。只在获得贷款的企业中观察贷款利率，就可能产生选择性观测问题。

## 1. 贷款金额为 0 与贷款利率 missing 的区别

设 $B_i$ 是企业银行贷款金额，$rate_i$ 是企业贷款利率。

对于没有获得贷款的企业：

$$
B_i=0
$$

这是一个可以观测到的结果。但同一家企业的贷款利率并不是：

$$
rate_i=0
$$

而是：

$$
rate_i \ \text{is missing}
$$

这一区别决定了模型选择。研究贷款金额时，可以考虑 Tobit 或 Two-part model；研究贷款利率时，如果利率只在获得贷款企业中可观测，就需要考虑 Heckman selection。

## 2. 信贷审批中的选择机制

银行贷款不是普通随机样本。企业是否能进入贷款利率样本，至少受到以下机制影响：

- 企业是否提出贷款申请；
- 银行是否愿意进入授信审查流程；
- 企业是否具有足够抵押能力；
- 银行服务可得性是否足够；
- 企业风险是否超过银行可接受范围。

如果这些选择因素还与贷款利率方程中的未观测因素相关，那么只在获得贷款企业中估计利率方程，就可能得到有偏结果。

## 3. 模型设定：选择方程与结果方程

设 $D_i=1$ 表示企业获得贷款，因此贷款利率可观测。选择方程为：

$$
D_i^*
=
\gamma_0
+\gamma_1 collateral_i
+\gamma_2 bank\_access_i
-\gamma_3 risk_i
+v_i
$$

$$
D_i=1(D_i^*>0)
$$

结果方程为：

$$
rate_i
=
\beta_0
-\beta_1 collateral_i
+\beta_2 risk_i
+u_i,
\quad D_i=1
$$

变量含义如下。

| 变量 | 选择方程 | 利率方程 | 经济含义 |
|---|---:|---:|---|
| `collateral` | 是 | 是 | 抵押能力提高获贷概率，也可能降低贷款利率 |
| `bank_access` | 是 | 否 | 主要影响企业能否进入银行信贷关系 |
| `risk` | 是 | 是 | 风险较高会降低获贷概率，同时提高贷款利率 |

这个设定故意保持简洁。重点不是列出尽可能多的控制变量，而是讲清楚选择机制和结果机制的差别。

![Heckman selection 的选择性观测机制](./figs/limit_dep_heckman_fig01_selection_mechanism.png)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

df = pd.read_csv("./data/heckman_credit_sim.csv")
df[["D", "loan_rate", "collateral", "bank_access", "risk"]].describe().round(3)

## 4. 为什么 naive OLS 可能有偏？

研究者实际能估计的是：

$$
E(rate_i\mid x_i,D_i=1)
$$

而想理解的通常是：

$$
E(rate_i\mid x_i)
$$

如果进入贷款利率样本的选择机制与结果方程误差项相关，即：

$$
Cov(u_i,v_i)\ne 0
$$

那么在 $D_i=1$ 的样本中：

$$
E(u_i\mid D_i=1)\ne 0
$$

这时，直接在获得贷款企业样本上估计：

$$
rate_i=x_i'\beta+u_i
$$

就会遗漏一个由选择机制产生的条件均值项。

![总体潜在贷款利率与观测贷款利率](./figs/limit_dep_heckman_fig02_observed_rate_sample.png)

图中总体潜在贷款利率与实际可观测贷款利率分布不同。原因是只有获得贷款的企业才进入贷款利率样本。

## 5. inverse Mills ratio 的来源

选择方程为：

$$
D_i^*=z_i'\gamma+v_i
$$

$$
D_i=1(D_i^*>0)
$$

在获得贷款样本中：

$$
D_i=1
\iff
v_i>-z_i'\gamma
$$

如果 $(u_i,v_i)$ 联合正态，则：

$$
E(u_i\mid D_i=1,z_i)
=
\rho\sigma_u
\lambda(z_i'\gamma)
$$

其中：

$$
\lambda(z_i'\gamma)
=
\frac{\phi(z_i'\gamma)}{\Phi(z_i'\gamma)}
$$

这就是 inverse Mills ratio (IMR)。将其加入结果方程，可写为：

$$
rate_i
=
x_i'\beta
+
\rho\sigma_u \lambda(z_i'\gamma)
+
\epsilon_i,
\quad D_i=1
$$

## 6. Heckman two-step

Heckman two-step 的流程是：

1. **Step 1**: 使用全部样本估计选择方程，得到 $\hat\gamma$；
2. **Step 2**: 对 $D_i=1$ 的样本计算 $\hat\lambda_i$；
3. **Step 3**: 在观测到贷款利率的样本中估计：

$$
rate_i
=
x_i'\beta
+
\delta \hat\lambda_i
+
\epsilon_i
$$

如果 $\delta$ 显著，说明选择项与贷款利率方程相关。更重要的是，加入 $\hat\lambda_i$ 后，$x_i$ 的系数解释的是控制选择性观测后的利率关系。

In [ ]:
# ============================================================
# Heckman two-step：简化实现
# ============================================================

# Step 1：选择方程
Z = sm.add_constant(df[["collateral", "bank_access", "risk"]])
probit = sm.Probit(df["D"], Z).fit(disp=False)

# 计算 IMR
xb = probit.predict(Z, linear=True)
df["imr"] = stats.norm.pdf(xb) / np.maximum(stats.norm.cdf(xb), 1e-12)

# Step 2：只在利率可观测样本中估计结果方程
sel = df["D"] == 1

X_naive = sm.add_constant(df.loc[sel, ["collateral", "risk"]])
naive = sm.OLS(df.loc[sel, "loan_rate"], X_naive).fit()

X_heck = sm.add_constant(df.loc[sel, ["collateral", "risk", "imr"]])
heckman_2s = sm.OLS(df.loc[sel, "loan_rate"], X_heck).fit()

pd.DataFrame({
    "Naive OLS": naive.params,
    "Heckman two-step": heckman_2s.params
}).round(4)

![IMR 与观测贷款利率](./figs/limit_dep_heckman_fig03_imr_correction.png)

## 7. 排他性变量如何理解？

Heckman 模型最好有一个进入选择方程但不直接进入结果方程的变量，也就是排他性变量。这里的候选变量是 `bank_access`。

在本章案例中，`bank_access` 可以理解为企业附近银行网点、银企关系便利度或本地信贷服务可得性。它可以影响企业是否获得贷款，因为银行服务可得性越高，企业越容易进入授信流程。

但它不一定直接决定贷款利率。贷款利率主要由企业风险、抵押能力、期限结构和银行定价规则决定。因此，在一个简化教学例子中，可以把 `bank_access` 作为选择方程中的排他性变量候选。

这个设定必须依靠经济机制论证。不能仅仅因为 `bank_access` 在选择方程中显著，就把它当作合格的排他性变量。如果银行可得性同时改变银行竞争程度、议价能力或贷款利率优惠，那么排他性限制就需要重新讨论。

## 8. Two-part model 与 Heckman 的区别

| 问题 | Two-part model | Heckman selection |
|---|---|---|
| 未参与者的结果 | 结果为 0 | 结果不可观测 |
| 典型因变量 | 贷款金额 | 贷款利率 |
| 样本是否都在数据中 | 是 | 企业在数据中，但结果变量只对部分样本可见 |
| 核心问题 | 真实 0 的建模 | 选择性观测导致的偏误 |
| 关键设定 | $E(B_i\mid z_i,x_i)=p_i\mu_i^+$ | $E(u_i\mid D_i=1)\ne 0$ |

这也是为什么企业贷款金额和企业贷款利率虽然来自同一信贷背景，却对应不同模型。没有贷款时，贷款金额可以是 0；没有贷款时，贷款利率不是 0，而是不存在。

## 9. 小结

Heckman selection 的核心不是“样本少了”，而是结果变量只在非随机选择后的样本中可观测。本章最重要的三个判断是：

- 没有贷款企业的贷款利率是 missing，不是 0；
- 选择方程解释企业是否进入贷款利率样本；
- 结果方程解释在贷款利率可观测样本中的利率水平。

如果能把选择机制、结果机制和排他性变量的经济含义讲清楚，Heckman 模型就不再只是一个机械的两阶段命令。